# Customer Segmentation Analysis

**Oasis Infobyte Data Analytics Internship — Level 1, Task 2**

This notebook uses RFM analysis and K-Means clustering to segment e-commerce customers based on purchasing behaviour.


## 1. Import Libraries


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

sns.set_theme(style="whitegrid")
np.random.seed(42)


## 2. Load the UCI Online Retail Dataset

The dataset is retrieved from the UCI Machine Learning Repository using its dataset interface. This avoids committing the large raw Excel file to GitHub.


In [ ]:
from ucimlrepo import fetch_ucirepo

online_retail = fetch_ucirepo(id=352)
df = online_retail.data.features.copy()

print("Dataset shape:", df.shape)
df.head()


## 3. Initial Inspection


In [ ]:
print("Shape:", df.shape)
print("\nData types:")
display(df.dtypes.to_frame("dtype"))

print("\nMissing values:")
display(df.isna().sum().to_frame("missing_values"))

print("\nDuplicate rows:", df.duplicated().sum())

print("\nSummary statistics:")
display(df.describe(include="all").T)


## Observation

The initial inspection shows the structure and quality of the transaction-level data. CustomerID is required for customer-level segmentation, so transactions without a customer identifier cannot be assigned to a customer segment.


## 4. Data Cleaning


In [ ]:
data = df.copy()

data["InvoiceDate"] = pd.to_datetime(data["InvoiceDate"], errors="coerce")

before_rows = len(data)

data = data.dropna(subset=["CustomerID", "InvoiceDate"])
data = data[data["Quantity"] > 0]
data = data[data["UnitPrice"] > 0]
data = data.drop_duplicates()

data["CustomerID"] = data["CustomerID"].astype(int)
data["TotalAmount"] = data["Quantity"] * data["UnitPrice"]

after_rows = len(data)

print("Rows before cleaning:", before_rows)
print("Rows after cleaning:", after_rows)
print("Rows removed:", before_rows - after_rows)

display(data.head())


## Observation

Missing customer IDs are removed because RFM segmentation requires a customer identifier. Non-positive quantities represent returns or cancellations and are excluded from positive-purchase segmentation. Transactions with non-positive prices are also excluded.


## 5. Customer-Level RFM Features

RFM represents three core purchasing behaviours:

- **Recency:** how recently the customer purchased.
- **Frequency:** how often the customer purchased.
- **Monetary:** how much the customer spent.


In [ ]:
reference_date = data["InvoiceDate"].max() + pd.Timedelta(days=1)

rfm = data.groupby("CustomerID").agg(
    Recency=("InvoiceDate", lambda x: (reference_date - x.max()).days),
    Frequency=("InvoiceNo", "nunique"),
    Monetary=("TotalAmount", "sum")
).reset_index()

rfm["AveragePurchaseValue"] = rfm["Monetary"] / rfm["Frequency"]

print("Number of customers:", len(rfm))
display(rfm.describe().T)


## 6. RFM Distribution


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

sns.histplot(rfm["Recency"], kde=True, ax=axes[0])
axes[0].set_title("Recency Distribution")

sns.histplot(rfm["Frequency"], kde=True, ax=axes[1])
axes[1].set_title("Frequency Distribution")

sns.histplot(rfm["Monetary"], kde=True, ax=axes[2])
axes[2].set_title("Monetary Distribution")

plt.tight_layout()
plt.show()


## Observation

RFM variables can be strongly right-skewed because a relatively small number of customers may purchase very frequently or spend substantially more than the typical customer. A log transformation helps reduce the influence of extreme values before clustering.


## 7. Prepare Features for K-Means


In [ ]:
rfm_model = rfm[["Recency", "Frequency", "Monetary"]].copy()

rfm_model["Recency"] = np.log1p(rfm_model["Recency"])
rfm_model["Frequency"] = np.log1p(rfm_model["Frequency"])
rfm_model["Monetary"] = np.log1p(rfm_model["Monetary"])

scaler = StandardScaler()
X = scaler.fit_transform(rfm_model)

print("Feature matrix shape:", X.shape)


## 8. Elbow Method

The Elbow Method compares K-Means inertia across different values of K. The point where the reduction in inertia begins to level off provides a practical candidate for the number of clusters.


In [ ]:
inertias = []
k_values = range(2, 9)

for k in k_values:
    model = KMeans(n_clusters=k, random_state=42, n_init=20)
    model.fit(X)
    inertias.append(model.inertia_)

plt.figure(figsize=(9, 5))
plt.plot(list(k_values), inertias, marker="o")
plt.title("Elbow Method for K-Means")
plt.xlabel("Number of Clusters (K)")
plt.ylabel("Inertia")
plt.xticks(list(k_values))
plt.show()


## Select K

Review the elbow curve above and select the K value where the curve begins to flatten. The default below uses **K = 4** as a practical starting point for four actionable customer profiles. If the executed elbow chart clearly indicates another value, change `optimal_k` before the final clustering run.


In [ ]:
optimal_k = 4

kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=20)
rfm["Cluster"] = kmeans.fit_predict(X)

print("Selected K:", optimal_k)
display(rfm["Cluster"].value_counts().sort_index().to_frame("Customers"))


## 9. Cluster Profiles


In [ ]:
cluster_profile = rfm.groupby("Cluster").agg(
    Customers=("CustomerID", "count"),
    Avg_Recency=("Recency", "mean"),
    Avg_Frequency=("Frequency", "mean"),
    Avg_Monetary=("Monetary", "mean"),
    Avg_Purchase_Value=("AveragePurchaseValue", "mean")
).round(2)

cluster_profile


## 10. Assign Business-Friendly Segment Names

The names below are assigned from the RFM profile rather than from the raw cluster number. Lower Recency and higher Frequency/Monetary generally indicate stronger engagement.


In [ ]:
profile = cluster_profile.copy()

profile["RecencyRank"] = profile["Avg_Recency"].rank(ascending=True)
profile["FrequencyRank"] = profile["Avg_Frequency"].rank(ascending=False)
profile["MonetaryRank"] = profile["Avg_Monetary"].rank(ascending=False)
profile["ValueIndex"] = profile[["RecencyRank", "FrequencyRank", "MonetaryRank"]].mean(axis=1)

ranked = profile.sort_values("ValueIndex")

segment_names = ["High-Value / Loyal", "Active Potential", "Regular Customers", "At-Risk / Inactive"]
segment_names = segment_names[:len(ranked)]

segment_map = dict(zip(ranked.index, segment_names))
rfm["Segment"] = rfm["Cluster"].map(segment_map)

segment_profile = rfm.groupby("Segment").agg(
    Customers=("CustomerID", "count"),
    Avg_Recency=("Recency", "mean"),
    Avg_Frequency=("Frequency", "mean"),
    Avg_Monetary=("Monetary", "mean"),
    Avg_Purchase_Value=("AveragePurchaseValue", "mean")
).sort_values("Avg_Monetary", ascending=False).round(2)

segment_profile


## 11. Cluster Visualization — Recency vs Monetary


In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=rfm,
    x="Recency",
    y="Monetary",
    hue="Segment",
    palette="tab10",
    alpha=0.7
)
plt.title("Customer Segments: Recency vs Monetary")
plt.xlabel("Recency (days)")
plt.ylabel("Monetary Value")
plt.legend(title="Segment", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()


## 12. Cluster Visualization — Frequency vs Monetary


In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=rfm,
    x="Frequency",
    y="Monetary",
    hue="Segment",
    palette="tab10",
    alpha=0.7
)
plt.title("Customer Segments: Frequency vs Monetary")
plt.xlabel("Purchase Frequency")
plt.ylabel("Monetary Value")
plt.legend(title="Segment", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()


## 13. Customers per Segment


In [ ]:
segment_counts = rfm["Segment"].value_counts()

plt.figure(figsize=(9, 5))
sns.barplot(x=segment_counts.index, y=segment_counts.values)
plt.title("Number of Customers per Segment")
plt.xlabel("Customer Segment")
plt.ylabel("Number of Customers")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

display(segment_counts.to_frame("Customers"))


## 14. Marketing Recommendations


In [ ]:
recommendations = {
    "High-Value / Loyal": "Prioritize retention with VIP rewards, early access, premium offers and personalized loyalty benefits.",
    "Active Potential": "Use cross-selling, bundles and loyalty incentives to increase purchase frequency and customer value.",
    "Regular Customers": "Use personalized promotions, product recommendations and repeat-purchase incentives.",
    "At-Risk / Inactive": "Run win-back campaigns, reminders and targeted reactivation offers with limited-time incentives."
}

recommendation_table = pd.DataFrame({
    "Segment": list(recommendations.keys()),
    "Recommended Marketing Action": list(recommendations.values())
})

display(recommendation_table)


## 15. Export Results


In [ ]:
final_summary = segment_profile.copy()
final_summary["Recommended Action"] = final_summary.index.map(recommendations)

display(final_summary)

rfm.to_csv("customer_segments_rfm.csv", index=False)
final_summary.to_csv("cluster_profile_summary.csv")

print("Saved: customer_segments_rfm.csv")
print("Saved: cluster_profile_summary.csv")


## Conclusion

RFM analysis converts transaction-level e-commerce data into customer-level behavioural features. K-Means clustering then groups customers with similar purchasing patterns after the RFM variables are standardized.

The resulting segments can be used to differentiate retention, cross-selling, loyalty and reactivation strategies. The exact segment sizes and profiles should always be read from the executed cluster summary rather than assumed in advance.
